# YZM304 Proje Odevi 1

Bu notebook, laboratuvarda kullanilan tek gizli katmanli MLP akisina sadik kalarak farkli bir veri seti uzerinde manuel ve scikit-learn tabanli siniflandirma deneylerini tekrarlar.

## 1. Kurulum ve Amac

- Veri seti: `sklearn.datasets.load_breast_cancer`
- Problem tipi: ikili siniflandirma
- Referans model: laboratuvardaki tek gizli katmanli manuel MLP
- Gelistirme hedefi: on isleme, daha derin mimari, mini-batch ve L2 regularization etkisini incelemek

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_breast_cancer_dataframe, prepare_dataset
from src.models.manual_mlp import ManualMLPClassifier
from src.models.sklearn_mlp import train_sklearn_mlp
from src.reporting import evaluate_predictions

ROOT = Path.cwd()
OUTPUTS = ROOT / 'outputs'
plt.rcParams['figure.figsize'] = (10, 4)

## 2. Veri Setini Inceleme

In [ ]:
df = load_breast_cancer_dataframe()
print(df.info())
df.head()

In [ ]:
df.describe().T.head(10)

In [ ]:
df.isnull().sum()

In [ ]:
target_counts = df.iloc[:, -1].value_counts().sort_index()
target_counts

## 3. Veri Hazirlama

Odevde overfitting ve underfitting incelemesi istendigi icin `train/validation/test` ayirimi kullanildi.

In [ ]:
dataset_raw = prepare_dataset(standardize=False)
dataset_std = prepare_dataset(standardize=True)

print('Raw train:', dataset_raw.X_train.shape, dataset_raw.y_train.shape)
print('Raw val:', dataset_raw.X_val.shape, dataset_raw.y_val.shape)
print('Raw test:', dataset_raw.X_test.shape, dataset_raw.y_test.shape)
print('Standardized train:', dataset_std.X_train.shape, dataset_std.y_train.shape)

## 4. Manuel MLP Fonksiyonlari

Ayni notebookta tum sinifi tekrar yazmak yerine, ayni mantigi koruyan kod `src/models/manual_mlp.py` dosyasina tasindi. Boylece gereksiz fonksiyon tekrari azaltildi.

In [ ]:
def run_manual_experiment(name, dataset, hidden_layers, n_steps, learning_rate, l2_lambda, batch_size=None):
    model = ManualMLPClassifier(
        input_dim=dataset.X_train.shape[1],
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        l2_lambda=l2_lambda,
        batch_size=batch_size,
        seed=42,
    )
    history = model.fit(
        dataset.X_train,
        dataset.y_train,
        X_val=dataset.X_val,
        y_val=dataset.y_val,
        n_steps=n_steps,
    )
    metrics = evaluate_predictions(dataset.y_test, model.predict(dataset.X_test))
    metrics['validation_accuracy'] = history.val_accuracy[-1]
    metrics['train_accuracy'] = history.train_accuracy[-1]
    return model, history, metrics

## 5. Laboratuvar Referans Modeli

In [ ]:
manual_lab_model, manual_lab_history, manual_lab_metrics = run_manual_experiment(
    name='manual_lab_baseline',
    dataset=dataset_raw,
    hidden_layers=(6,),
    n_steps=500,
    learning_rate=0.01,
    l2_lambda=0.0,
)
manual_lab_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(manual_lab_history.train_loss, label='Train Loss')
axes[0].plot(manual_lab_history.val_loss, label='Validation Loss')
axes[0].set_title('Lab Baseline Loss')
axes[0].legend()
axes[1].plot(manual_lab_history.train_accuracy, label='Train Accuracy')
axes[1].plot(manual_lab_history.val_accuracy, label='Validation Accuracy')
axes[1].set_title('Lab Baseline Accuracy')
axes[1].legend()
plt.show()

Bu sonuc, yeni veri setinde laboratuvardaki hiperparametrelerin underfitting urettigini gosterir. Bu durum odevde istenen bias/variance tartismasi icin referans deney olarak tutuldu.

## 6. Iyilestirilmis Manuel Modeller

In [ ]:
manual_improved_model, manual_improved_history, manual_improved_metrics = run_manual_experiment(
    name='manual_improved_baseline',
    dataset=dataset_std,
    hidden_layers=(6,),
    n_steps=1000,
    learning_rate=0.05,
    l2_lambda=0.0,
)

manual_deep_model, manual_deep_history, manual_deep_metrics = run_manual_experiment(
    name='manual_deeper_regularized',
    dataset=dataset_std,
    hidden_layers=(12, 6),
    n_steps=1500,
    learning_rate=0.05,
    l2_lambda=0.001,
    batch_size=32,
)

pd.DataFrame([
    {'model': 'manual_lab_baseline', **manual_lab_metrics},
    {'model': 'manual_improved_baseline', **manual_improved_metrics},
    {'model': 'manual_deeper_regularized', **manual_deep_metrics},
])[['model', 'accuracy', 'precision', 'recall', 'f1', 'validation_accuracy', 'train_accuracy']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(manual_improved_history.train_loss, label='Improved Train Loss')
axes[0].plot(manual_improved_history.val_loss, label='Improved Val Loss')
axes[0].plot(manual_deep_history.train_loss, label='Deep Train Loss')
axes[0].plot(manual_deep_history.val_loss, label='Deep Val Loss')
axes[0].set_title('Manual Model Loss Curves')
axes[0].legend()
axes[1].plot(manual_improved_history.train_accuracy, label='Improved Train Accuracy')
axes[1].plot(manual_improved_history.val_accuracy, label='Improved Val Accuracy')
axes[1].plot(manual_deep_history.train_accuracy, label='Deep Train Accuracy')
axes[1].plot(manual_deep_history.val_accuracy, label='Deep Val Accuracy')
axes[1].set_title('Manual Model Accuracy Curves')
axes[1].legend()
plt.show()

## 7. Scikit-learn MLPClassifier Karsilastirmasi

Ayni problem, benzer mimariler ve `SGD` solver ile `MLPClassifier` kullanilarak tekrar egitildi.

In [ ]:
sklearn_baseline = train_sklearn_mlp(
    dataset_std.X_train,
    dataset_std.y_train,
    hidden_layers=(6,),
    learning_rate=0.05,
    alpha=0.0,
    batch_size=len(dataset_std.X_train),
    max_iter=1000,
    random_state=42,
)
sklearn_deep = train_sklearn_mlp(
    dataset_std.X_train,
    dataset_std.y_train,
    hidden_layers=(12, 6),
    learning_rate=0.05,
    alpha=0.001,
    batch_size=32,
    max_iter=1500,
    random_state=42,
)

sklearn_baseline_metrics = evaluate_predictions(dataset_std.y_test, sklearn_baseline.model.predict(dataset_std.X_test).reshape(-1, 1))
sklearn_deep_metrics = evaluate_predictions(dataset_std.y_test, sklearn_deep.model.predict(dataset_std.X_test).reshape(-1, 1))

pd.DataFrame([
    {'model': 'sklearn_baseline', **sklearn_baseline_metrics},
    {'model': 'sklearn_deeper_regularized', **sklearn_deep_metrics},
])[['model', 'accuracy', 'precision', 'recall', 'f1']]

## 8. Kaydedilen Sonuclari Oku

In [ ]:
results_path = OUTPUTS / 'experiment_results.json'
saved_results = json.loads(results_path.read_text(encoding='utf-8'))
pd.DataFrame([
    {'group': 'manual', 'model': item['config']['name'], **{k: item['metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1']}}
    for item in saved_results['manual_experiments']
] + [
    {'group': 'sklearn', 'model': item['config']['name'], **{k: item['metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1']}}
    for item in saved_results['sklearn_experiments']
])

## 9. Sonuc

- Notebook, laboratuvar notebookundaki cekirdek akisi korur.
- Farkli veri seti ile ayni model once referans olarak kurulmustur.
- Daha sonra standardization, daha uzun egitim, ikinci gizli katman, mini-batch ve L2 regularization eklenmistir.
- Son durumda en iyi manuel model `manual_deeper_regularized`, en iyi dogruluk ise `0.9386` olarak elde edilmistir.